# EEND-SS Training

**Before running:**
1. Runtime → Change runtime type → **T4 GPU**
2. Upload `processed.zip` to `MyDrive/voice_isolation_desktop/`
3. Upload all `src/` code files to `MyDrive/voice_isolation_desktop/src/`
4. Run cells in order

**Drive structure needed:**
```
MyDrive/voice_isolation_desktop/
    processed.zip          ← your dataset zip
    src/
        models/            ← conv_tasnet.py, eend.py, eend_ss.py
        training/          ← losses.py, trainer.py, config.py
    models/                ← create this empty folder (checkpoints go here)
    outputs/               ← create this empty folder (logs go here)
```

In [2]:
# ── CELL 1: Check GPU ─────────────────────────────────────────────────────
import torch
print('CUDA available :', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU            :', torch.cuda.get_device_name(0))
    print('GPU memory     :', round(torch.cuda.get_device_properties(0).total_memory/1e9, 1), 'GB')
else:
    print('NO GPU — go to Runtime > Change runtime type > T4 GPU')
print('PyTorch        :', torch.__version__)

import subprocess
subprocess.run(['pip', 'install', 'torchaudio', '--quiet'], check=True)
print('torchaudio     : ready')

CUDA available : True
GPU            : NVIDIA RTX A6000
GPU memory     : 51.5 GB
PyTorch        : 2.5.1+cu121
torchaudio     : ready


In [ ]:
# ── CELL 2: Mount Drive ───────────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

import os
from pathlib import Path

DRIVE_ROOT = '/content/drive/MyDrive/voice_isolation_desktop'
ZIP_PATH   = f'{DRIVE_ROOT}/processed.zip'

assert os.path.exists(DRIVE_ROOT), f'Folder not found: {DRIVE_ROOT}'
assert os.path.exists(ZIP_PATH),   f'Zip not found: {ZIP_PATH} — upload processed.zip first'

zip_size_gb = os.path.getsize(ZIP_PATH) / 1e9
print(f'Drive root  : {DRIVE_ROOT}')
print(f'processed.zip found: {zip_size_gb:.1f} GB')

In [ ]:
# ── CELL 3: Extract zip to local SSD ─────────────────────────────────────
# Unzips directly to /content/data/ on the fast local SSD.
# Drive -> SSD copy of a zip is much faster than copying individual files.
# Takes ~3-8 minutes depending on zip size.
# Skip this cell if you already ran it this session.

import zipfile, time, os
from pathlib import Path

ZIP_PATH  = '/content/drive/MyDrive/voice_isolation_desktop/processed.zip'
EXTRACT_TO = '/content/data'
DATA_ROOT  = '/content/data/processed'

# skip if already extracted
if Path(DATA_ROOT).exists() and Path(f'{DATA_ROOT}/train/train_manifest.json').exists():
    print('Already extracted — skipping')
    print(f'Data root: {DATA_ROOT}')
else:
    os.makedirs(EXTRACT_TO, exist_ok=True)
    print(f'Extracting {ZIP_PATH}')
    print('This takes 3-8 minutes...')
    t0 = time.time()

    with zipfile.ZipFile(ZIP_PATH, 'r') as zf:
        members = zf.namelist()
        total   = len(members)
        print(f'Total files in zip: {total:,}')

        # extract in chunks and print progress
        chunk = max(1, total // 20)   # print 20 progress updates
        for i, member in enumerate(members):
            zf.extract(member, EXTRACT_TO)
            if (i + 1) % chunk == 0:
                pct = (i + 1) / total * 100
                elapsed = time.time() - t0
                eta = elapsed / (i+1) * (total - i - 1)
                print(f'  {pct:.0f}% ({i+1:,}/{total:,}) — elapsed {elapsed:.0f}s — ETA {eta:.0f}s')

    elapsed = time.time() - t0
    print(f'\nExtraction complete in {elapsed:.0f}s ({elapsed/60:.1f} min)')

# verify extraction
for split, expected_manifest in [
    ('train', 'train_manifest.json'),
    ('val',   'val_manifest.json'),
    ('test',  'test_manifest.json'),
]:
    p = Path(DATA_ROOT) / split / expected_manifest
    wav_count = len(list((Path(DATA_ROOT)/split/'mixtures').glob('*.wav'))) \
                if (Path(DATA_ROOT)/split/'mixtures').exists() else 0
    npy_count = len(list((Path(DATA_ROOT)/split/'mixtures').glob('*.npy'))) \
                if (Path(DATA_ROOT)/split/'mixtures').exists() else 0
    status = '✓' if p.exists() else '✗'
    print(f'  {status} {split}: manifest={p.exists()} | {wav_count} wav | {npy_count} npy')

print(f'\nData root ready: {DATA_ROOT}')

In [ ]:
# ── CELL 4: Write src/ code files ────────────────────────────────────────
import os, shutil

os.makedirs('/content/src/data',     exist_ok=True)
os.makedirs('/content/src/models',   exist_ok=True)
os.makedirs('/content/src/training', exist_ok=True)

# __init__ files
open('/content/src/__init__.py','w').write('')
open('/content/src/data/__init__.py','w').write(
    'from .dataset import EENDSSDataset\n'
    'from .loaders import get_loaders\n'
    'from .collate import collate_train, collate_eval\n'
)
open('/content/src/models/__init__.py','w').write(
    'from .conv_tasnet import ConvTasNet, Encoder, Separator, Decoder, TCN\n'
    'from .eend import EEND, TransformerEncoderBlock, EncoderDecoderAttractor\n'
    'from .eend_ss import EENDSS\n'
)
open('/content/src/training/__init__.py','w').write(
    'from .losses import EENDSSLoss, si_sdr, pit_si_sdr_loss, pit_diarization_loss\n'
    'from .trainer import Trainer, get_device\n'
    'from .config import Config, get_config\n'
)

# copy model + training files from Drive
DRIVE_SRC = '/content/drive/MyDrive/voice_isolation_desktop/src'
for fname in ['conv_tasnet.py', 'eend.py', 'eend_ss.py']:
    shutil.copy(f'{DRIVE_SRC}/models/{fname}',   f'/content/src/models/{fname}')
    print(f'  copied models/{fname}')
for fname in ['losses.py', 'trainer.py', 'config.py']:
    shutil.copy(f'{DRIVE_SRC}/training/{fname}', f'/content/src/training/{fname}')
    print(f'  copied training/{fname}')

print('\nAll model files copied')

In [ ]:
# ── CELL 5: Write dataset.py + collate.py + loaders.py ───────────────────
open('/content/src/data/dataset.py','w').write('''
import json, random
from pathlib import Path
import numpy as np
import torch
import torchaudio
from torch.utils.data import Dataset

SAMPLE_RATE = 16_000
MAX_SPEAKERS = 3
LABEL_FRAME_RATE = 100

class EENDSSDataset(Dataset):
    def __init__(self, manifest_path, chunk_duration=4.0, mode="train", max_samples=None):
        self.manifest_path = Path(manifest_path)
        self.chunk_samples = int(chunk_duration * SAMPLE_RATE)
        self.chunk_frames  = int(chunk_duration * LABEL_FRAME_RATE)
        self.training = mode == "train"
        with open(self.manifest_path) as f:
            self.samples = json.load(f)
        if max_samples is not None:
            self.samples = self.samples[:max_samples]
        print(f"[EENDSSDataset] {mode}: {len(self.samples)} samples")

    def __len__(self):
        return len(self.samples)

    def resolve(self, path_str):
        """Remap any absolute path to local SSD data root."""
        p = Path(path_str)
        # data root = manifest parent parent (data/processed/)
        data_root = self.manifest_path.parent.parent
        if p.is_absolute():
            # find processed/ in the path and take everything after it
            parts = p.parts
            try:
                idx = next(i for i,x in enumerate(parts) if x=="processed")
                return data_root / Path(*parts[idx+1:])
            except StopIteration:
                return p
        return data_root / path_str

    def __getitem__(self, idx):
        meta = self.samples[idx]
        mixture, sr = torchaudio.load(self.resolve(meta["mixture_path"]))
        if mixture.shape[0] > 1:
            mixture = mixture.mean(dim=0, keepdim=True)

        source_list = []
        for path in meta["source_paths"]:
            src, _ = torchaudio.load(self.resolve(path))
            if src.shape[0] > 1: src = src.mean(dim=0, keepdim=True)
            source_list.append(src.squeeze(0))

        labels = torch.from_numpy(np.load(self.resolve(meta["label_path"]))).float()
        num_speakers  = meta["num_speakers"]
        total_samples = mixture.shape[-1]
        total_frames  = labels.shape[0]

        if self.training:
            mixture, source_list, labels = self._chunk(
                mixture, source_list, labels, total_samples, total_frames)

        T = mixture.shape[-1]
        T_frames = labels.shape[0]
        padded_sources = torch.zeros(MAX_SPEAKERS, T)
        for i, src in enumerate(source_list):
            src_len = min(src.shape[0], T)
            padded_sources[i, :src_len] = src[:src_len]
        padded_labels = torch.zeros(T_frames, MAX_SPEAKERS)
        padded_labels[:, :num_speakers] = labels

        return {"mixture": mixture, "sources": padded_sources,
                "labels": padded_labels, "num_speakers": num_speakers,
                "mixture_id": meta["mixture_id"]}

    def _chunk(self, mixture, source_list, labels, total_samples, total_frames):
        max_start = total_samples - self.chunk_samples
        if max_start <= 0:
            pad = self.chunk_samples - total_samples
            mixture = torch.nn.functional.pad(mixture, (0, pad))
            source_list = [torch.nn.functional.pad(s,(0,pad)) for s in source_list]
            frame_pad = self.chunk_frames - total_frames
            if frame_pad > 0:
                labels = torch.nn.functional.pad(labels.T,(0,frame_pad)).T
            return mixture, source_list, labels[:self.chunk_frames]
        start_sample = random.randint(0, max_start)
        start_frame  = int(start_sample / SAMPLE_RATE * LABEL_FRAME_RATE)
        end_frame    = min(start_frame + self.chunk_frames, total_frames)
        mixture      = mixture[:, start_sample:start_sample+self.chunk_samples]
        source_list  = [s[start_sample:start_sample+self.chunk_samples] for s in source_list]
        labels       = labels[start_frame:end_frame]
        if labels.shape[0] < self.chunk_frames:
            pad = self.chunk_frames - labels.shape[0]
            labels = torch.nn.functional.pad(labels.T,(0,pad)).T
        return mixture, source_list, labels
''')

open('/content/src/data/collate.py','w').write('''
import torch
from torch.nn.utils.rnn import pad_sequence

def collate_train(batch):
    return {
        "mixture":      torch.stack([b["mixture"]  for b in batch]),
        "sources":      torch.stack([b["sources"]  for b in batch]),
        "labels":       torch.stack([b["labels"]   for b in batch]),
        "num_speakers": torch.tensor([b["num_speakers"] for b in batch]),
        "mixture_id":   [b["mixture_id"] for b in batch],
    }

def collate_eval(batch):
    batch     = sorted(batch, key=lambda b: b["mixture"].shape[-1], reverse=True)
    mixtures  = [b["mixture"].squeeze(0) for b in batch]
    sources   = [b["sources"] for b in batch]
    labels    = [b["labels"]  for b in batch]
    lengths   = torch.tensor([m.shape[-1] for m in mixtures])
    lengths_f = torch.tensor([l.shape[0]  for l in labels])
    num_spk   = torch.tensor([b["num_speakers"] for b in batch])
    mix_padded = pad_sequence(mixtures, batch_first=True).unsqueeze(1)
    T_max = mix_padded.shape[-1]
    src_padded = torch.zeros(len(batch), 3, T_max)
    for i, src in enumerate(sources):
        src_padded[i,:,:src.shape[-1]] = src
    T_f_max = max(l.shape[0] for l in labels)
    lbl_padded = torch.zeros(len(batch), T_f_max, 3)
    for i, lbl in enumerate(labels):
        lbl_padded[i,:lbl.shape[0],:] = lbl
    return {"mixture": mix_padded, "sources": src_padded, "labels": lbl_padded,
            "num_speakers": num_spk, "lengths": lengths, "lengths_f": lengths_f,
            "mixture_id": [b["mixture_id"] for b in batch]}
''')

open('/content/src/data/loaders.py','w').write('''
from pathlib import Path
from torch.utils.data import DataLoader
from .dataset import EENDSSDataset
from .collate import collate_train, collate_eval

def get_loaders(cfg):
    data_dir = Path(cfg["data_dir"])
    train_ds = EENDSSDataset(data_dir/"train"/"train_manifest.json",
        chunk_duration=cfg.get("chunk_duration",4.0), mode="train",
        max_samples=cfg.get("max_train",None))
    val_ds = EENDSSDataset(data_dir/"val"/"val_manifest.json",
        chunk_duration=cfg.get("chunk_duration",4.0), mode="val",
        max_samples=cfg.get("max_val",None))
    test_ds = EENDSSDataset(data_dir/"test"/"test_manifest.json",
        chunk_duration=cfg.get("chunk_duration",4.0), mode="test")
    nw = cfg.get("num_workers", 2)
    bs = cfg.get("batch_size", 8)
    train_loader = DataLoader(train_ds, batch_size=bs, shuffle=True,
        num_workers=nw, collate_fn=collate_train, pin_memory=True, drop_last=True)
    val_loader = DataLoader(val_ds, batch_size=4, shuffle=False,
        num_workers=nw, collate_fn=collate_eval, pin_memory=True)
    test_loader = DataLoader(test_ds, batch_size=4, shuffle=False,
        num_workers=nw, collate_fn=collate_eval, pin_memory=True)
    print(f"[loaders] train={len(train_ds)} val={len(val_ds)} test={len(test_ds)}")
    print(f"[loaders] train batches/epoch: {len(train_loader)}")
    return train_loader, val_loader, test_loader
''')
print('dataset.py + collate.py + loaders.py written')

In [ ]:
# ── CELL 6: Fix trainer.py (remove verbose kwarg) ─────────────────────────
# Newer PyTorch removed 'verbose' from ReduceLROnPlateau
content = open('/content/src/training/trainer.py').read()
content = content.replace(
    'patience=cfg.lr_patience,\n            verbose=True,',
    'patience=cfg.lr_patience,'
)
content = content.replace(
    'patience=cfg.lr_patience,\n            verbose=True\n',
    'patience=cfg.lr_patience,\n'
)
open('/content/src/training/trainer.py','w').write(content)
remaining = content.count('verbose')
print(f'trainer.py patched — verbose occurrences remaining: {remaining} (should be 0)')

In [ ]:
# ── CELL 7: Configure paths ───────────────────────────────────────────────
import sys
sys.path.insert(0, '/content')

import os
from src.training.config import Config

cfg = Config()
cfg.data_dir   = '/content/data/processed'      # local SSD — fast reads
cfg.output_dir = '/content/drive/MyDrive/voice_isolation_desktop/models'   # Drive — safe writes
cfg.log_dir    = '/content/drive/MyDrive/voice_isolation_desktop/outputs'  # Drive — safe writes
cfg.batch_size  = 8
cfg.num_workers = 2
cfg.max_val     = 200

os.makedirs(cfg.output_dir, exist_ok=True)
os.makedirs(cfg.log_dir,    exist_ok=True)

from pathlib import Path
assert Path(cfg.data_dir).exists(), f'Data SSD path missing — did Cell 3 finish? {cfg.data_dir}'
assert Path(f'{cfg.data_dir}/train/train_manifest.json').exists(), 'train manifest missing'
assert Path(f'{cfg.data_dir}/val/val_manifest.json').exists(),     'val manifest missing'

print('Config ready:')
print(f'  data_dir   : {cfg.data_dir}')
print(f'  output_dir : {cfg.output_dir}')
print(f'  log_dir    : {cfg.log_dir}')
print(f'  batch_size : {cfg.batch_size}')
print(f'  num_workers: {cfg.num_workers}')
print('\nAll paths verified ✓')

In [ ]:
# ── CELL 8: Smoke test (3 epochs, 200 samples) ────────────────────────────
from src.models.eend_ss import EENDSS
from src.training.trainer import Trainer
from src.data.loaders import get_loaders

cfg.max_train = 200
cfg.max_val   = 50
cfg.epochs    = 3
cfg.log_every = 5

model = EENDSS(
    N=cfg.N, L=cfg.L, B=cfg.B, H=cfg.H, P=cfg.P,
    X=cfg.X, R=cfg.R, C=cfg.C, d_model=cfg.d_model,
    n_heads=cfg.n_heads, n_layers=cfg.n_layers,
    d_ff=cfg.d_ff, dropout=cfg.dropout, subsample=cfg.subsample,
)

train_loader, val_loader, _ = get_loaders({
    'data_dir':       cfg.data_dir,
    'chunk_duration': cfg.chunk_duration,
    'batch_size':     cfg.batch_size,
    'num_workers':    cfg.num_workers,
    'max_train':      cfg.max_train,
    'max_val':        cfg.max_val,
})

trainer = Trainer(model, cfg)
trainer.train(train_loader, val_loader)
print('\nSmoke test PASSED ✓ — ready for full training')

In [ ]:
# ── CELL 9: FULL TRAINING ─────────────────────────────────────────────────
# Only run after smoke test passes.
# Reads from local SSD, saves checkpoints to Drive.
# Expected: ~10-15 min/epoch on T4, ~8-12 hours total.

from src.models.eend_ss import EENDSS
from src.training.trainer import Trainer
from src.data.loaders import get_loaders

cfg.max_train = None   # all 10k samples
cfg.max_val   = 200
cfg.epochs    = 50
cfg.log_every = 20

model = EENDSS(
    N=cfg.N, L=cfg.L, B=cfg.B, H=cfg.H, P=cfg.P,
    X=cfg.X, R=cfg.R, C=cfg.C, d_model=cfg.d_model,
    n_heads=cfg.n_heads, n_layers=cfg.n_layers,
    d_ff=cfg.d_ff, dropout=cfg.dropout, subsample=cfg.subsample,
)

train_loader, val_loader, _ = get_loaders({
    'data_dir':       cfg.data_dir,
    'chunk_duration': cfg.chunk_duration,
    'batch_size':     cfg.batch_size,
    'num_workers':    cfg.num_workers,
    'max_train':      None,
    'max_val':        200,
})

trainer = Trainer(model, cfg)
trainer.train(train_loader, val_loader)

In [ ]:
# ── CELL 10: Plot training curves ────────────────────────────────────────
import json, matplotlib.pyplot as plt
from pathlib import Path

history_path = Path(cfg.log_dir) / 'training_history.json'
h = json.loads(history_path.read_text())
epochs = range(1, len(h['train_loss'])+1)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, (tk, vk, title) in zip(axes, [
    ('train_loss',  'val_loss',  'Total Loss'),
    ('train_sisnr', 'val_sisnr', 'SI-SNR (dB) ↑'),
    ('train_diar',  'val_diar',  'Diarization Loss ↓'),
]):
    ax.plot(epochs, h[tk], label='Train')
    ax.plot(epochs, h[vk], label='Val')
    ax.set_title(title)
    ax.set_xlabel('Epoch')
    ax.legend()
    ax.grid(True)

plt.tight_layout()
out = str(Path(cfg.log_dir) / 'training_curves.png')
plt.savefig(out, dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved: {out}')
print(f'Epochs completed : {len(epochs)}')
print(f'Best val loss    : {min(h["val_loss"]):.4f}')
print(f'Best SI-SNR      : {max(h["val_sisnr"]):.2f} dB')